In [ ]:
#----------------------------#
# 0.) Idea
#----------------------------#
# Run PCA on
# Approaches: 
# 1.) Load Data
# 2.) Compute Embeddings PROBLEM: Don't know whether embeddings are vectors
# and whether I can compute the average
# 3.) PCA on subreddits
# 4.) PCA on submission of specific subreddits
# 5.) Multidimensional Scaling Tryout
# 6.) Vlads subreddits check
# Appendix 1: SVD/ PCA tests and sources
#
##CONCLUSION SO FAR: PCA over subreddits shows clustering, PCA over submissions within subreddits doesnt
#

import sys
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from collections import Counter
from IPython.core.display import display, HTML
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import import_ipynb

In [ ]:
home_path = %pwd
data_path = "/home/pavel/Documents/Keusch_Job/Data/"
plot_path  = "/home/pavel/Documents/Keusch_Job/Tasks/subreddit_decide/"
subroutines_path = "/home/pavel/Documents/Keusch_Job/Subroutines/"

In [ ]:
#Subroutines
%cd  $subroutines_path
%run "remove_contractions.py"
%run "basic_plots.ipynb"
%run "selected_subreddits.ipynb"
%run "data_loader.ipynb"
%cd  $home_path

In [ ]:
#----------------------------#
# 1.) Load Data
#----------------------------#

In [ ]:
S = load_data(data_name="SUBM_2020_01.csv", data_path = data_path)

In [ ]:
S  = S.merge(selected_subreddits[["subreddit","subr_id"]], how="inner",
                               left_on = "subreddit",right_on = "subreddit")
len(S)

In [ ]:
#Sample 
#S_complete = S.copy()
#S = S_complete.sample(n=10000, random_state=992747)

In [ ]:
#----------------------------#
# 2.) Compute Embeddings PROBLEM: Don't know whether embeddings are vectors
# and whether I can compute the average
#----------------------------#

In [ ]:
# WARNING: download through http link takes almost 30minutes
# Thats why I have it on disc in the data path
#module_url = "https://tfhub.dev/google/universal-sentence-encoder/4" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
#model = hub.load(module_url)
#thx to https://stackoverflow.com/questions/50322001/how-to-save-load-a-tensorflow-hub-module-to-from-a-custom-path
module_url = data_path + "universal-sentence-encoder_4"
embed_model = hub.load(module_url)

In [ ]:
def embed(input): return embed_model(input)
def tf_simil(s1,s2): return np.inner(s1,s2) # inner Product as similarity! cool, but why inner product? 
# wouldn't you take cosine? 
def tf_simil_matrix(embeddings, rd = 8): return [[round(tf_simil(s1,s2),rd) for s1 in embeddings] for s2 in embeddings]

In [ ]:
#test embeddings
s_1 = embed(["hello darkness my old friend"])
s_2 = embed(["hit me baby one more time"])
tf_simil(s_1,s_2)

In [ ]:
#create dictionary of embeddingns per post
def dictionary_from_submissions(df):
    #df: something like S, wiht an id and titles
    embed_dict = {}
    for i in range(0,len(df)):
        print(str(i) + " : " + df["id"].iloc[i])
        embed_dict[df["id"].iloc[i]] = embed([df["title"].iloc[i]])
    return(embed_dict)

In [ ]:
embed_dict = dictionary_from_submissions(S)

In [ ]:
#embed_dict = dictionary_from_submissions(S)

In [ ]:
# save dictionary to save time of computing. This here already took 5min
# thx to https://stackoverflow.com/questions/19201290/how-to-save-a-dictionary-to-a-file/32216025
np.save(data_path + 'embed_dict.npy', embed_dict)
#embed_dict = np.load('embed_dict.npy',allow_pickle='TRUE').item()

In [ ]:
def get_subreddit_embeddings(c_S):
    #c_S: current subreddit
    local_embed_list = []
    id_list = c_S["id"].unique().tolist()
    [local_embed_list.append(embed_dict[x]) for x in id_list]
    return(local_embed_list)

In [ ]:
def avg_embedding(list_of_embs):
    # averages of vectors should be allowed IF this is a vector
    # https://www.tf.uni-kiel.de/matwis/amat/admat_en/kap_2/basics/b2_1_1.html
    # https://math.stackexchange.com/questions/80923/average-of-multiple-vectors
    out = list_of_embs[0]
    for i in range(1,len(list_of_embs)):
        out = out + list_of_embs[i]
    return(out/len(list_of_embs))

In [ ]:
#WARNING: it seems like we don't have all the subreddits in S
print(len(selected_subreddits["subr_id"].unique().tolist()))
print(len(S["subr_id"].unique()))
print(len(list(set(subr_list) & set(S["subr_id"].unique()))))

In [ ]:
selected_subreddits = selected_subreddits.loc[selected_subreddits["subr_id"].isin(S["subr_id"]),].copy()

In [ ]:
subr_list = selected_subreddits["subr_id"].unique().tolist()

In [ ]:
# Extract average embedding per subreddit
subr_dict  = {}
for s in subr_list:
    current_subreddit = S.loc[S["subr_id"] == s,]
    current_embedding_list = get_subreddit_embeddings(current_subreddit)
    subr_dict[s] = avg_embedding(current_embedding_list)

In [ ]:
#----------------------------#
# 3.) PCA on subreddits
#----------------------------#
# many thx to https://towardsdatascience.com/entity-embedding-using-pca-and-kernel-pca-798b8b2e8c2f

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

In [ ]:
def subr_dict_2_matrix(s_dict, key_list):
    #s_dict: some dictionary where the values are embedings ()
    #key_list: a list of keys
    #RETURNS: the np.array matrix where each value is a row.
    emb_matrix = np.append(s_dict[key_list[0]], s_dict[key_list[1]],axis=0)
    for i in range(2,len(key_list)):
        emb_matrix = np.append(emb_matrix, s_dict[key_list[i]],axis=0)
    return(emb_matrix)

In [ ]:
subr_emb_matrix = subr_dict_2_matrix(s_dict = subr_dict, key_list = subr_list)

In [ ]:
# have to transpose because.... above the use the day dimensions as columns
subr_emb_matrix = np.transpose(subr_emb_matrix)
subr_emb_matrix.shape

In [ ]:
model = PCA(n_components=2).fit(subr_emb_matrix)

In [ ]:
%matplotlib inline
import seaborn as sns

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x'] + 0.0002, point['y'], str(point['val']), alpha = 0.5)

In [ ]:
def emb_plot(emb_matrix, plot_path = plot_path , file_name = "subreddits_avg_emb_PCA_labels",
             label_series = selected_subreddits["subreddit"], 
             legend = True, plot_labels= False, MDS = False):
    
    # PCA
    if MDS:
        category_vectors = MDS_embedding.fit_transform(emb_matrix.T)
    else:
        model = PCA(n_components=2).fit(emb_matrix)
        category_vectors = model.components_.T
    #    # transform df
    category_vector_frame=pd.DataFrame(category_vectors, index = label_series.tolist(), #index=subr_list, 
                                   columns=['col1', 'col2']).reset_index()
    #
    #Plot 
    plt.figure(figsize=(10, 10))
    if legend:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2',hue='index');
        plt.legend(loc='upper right')
    else:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2');
    if plot_labels:
        # need to do it because https://stackoverflow.com/questions/40339886/pandas-concat-generates-nan-values
        label_series_2 = label_series.copy()
        label_series_2.reset_index(drop=True, inplace=True)
        label_point(category_vector_frame.col1, category_vector_frame.col2,label_series_2, plt.gca())
    plt.savefig(fname=plot_path + file_name)

In [ ]:
emb_plot(subr_emb_matrix,  plot_path = plot_path , file_name = "subreddits_avg_emb_PCA_labels",
        label_series = selected_subreddits["subreddit"], legend = False, plot_labels= True)

In [ ]:
emb_plot(subr_emb_matrix,  plot_path = plot_path , file_name = "subreddits_avg_emb_PCA",
        label_series = selected_subreddits["subreddit"], legend = False, plot_labels= False)

In [ ]:
#CONCLUSION: Semantic Space of chosen subreddits is quite bundled
# We have to consider the withing-forum distribution: Words like "cuck" won't be an flag in some forums

In [ ]:
#----------------------------#
# 4.) PCA on submission of specific subreddits
#----------------------------#

In [ ]:
subs_grouped = selected_subreddits.merge(S.groupby(["subr_id"]).size().reset_index(), how = "inner", 
                          left_on = "subr_id", right_on = "subr_id")
subs_grouped.columns = ["subreddit","keep","category","language","subscribers_K",
                       "avg_ncomments","perc_removed","04-11","04-12","04-13","total",
                      "total7d","new","subr_id", "submission_count"]
subs_grouped[["subreddit","submission_count"]]
#Lets compare TrueReddit Something that may not be political at all with, Positive_News

In [ ]:
test_subr_id = selected_subreddits.loc[selected_subreddits["subreddit"]== "Coronavirus", "subr_id"].tolist()[0]
print(test_subr_id)
test_subr_posts = S.loc[S["subr_id"] == test_subr_id,]
test_post_list = test_subr_posts["id"].tolist()
test_post_embs = get_subreddit_embeddings(test_subr_posts)

In [ ]:
#WARNING: müsste es nicht post_list = subr_posts["id"].tolist() sein,
# und nicht post_list = subr_posts["subr_id"].tolist() ?
# Ja aber ist egal
def subreddit_emb_matrix_get(sub_name):   
    # extract data structures
    subr_id = selected_subreddits.loc[selected_subreddits["subreddit"]== sub_name, "subr_id"].tolist()[0]
    subr_posts = S.loc[S["subr_id"] == subr_id,]
    #post_list = subr_posts["id"].tolist()
    post_embs = get_subreddit_embeddings(subr_posts)
    # get folded matrix
    emb_matrix = np.append(post_embs[0], post_embs[1],axis=0)
    for i in range(2,len(post_embs)):
        emb_matrix = np.append(emb_matrix, post_embs[i],axis=0)
    return(emb_matrix,subr_posts)

In [ ]:
del pn_embs

In [ ]:
conservative_embs, conservative_posts = subreddit_emb_matrix_get("conservatives")
conservative_embs = np.transpose(conservative_embs)
conservative_embs.shape

In [ ]:
corona_embs, corona_posts = subreddit_emb_matrix_get("Coronavirus")
corona_embs = np.transpose(corona_embs)
corona_embs.shape

In [ ]:
emb_plot(conservative_embs,  plot_path = plot_path , file_name = "Submissions_conservatives_PCA",
        label_series = conservative_posts["id"], legend = False, plot_labels=False)

In [ ]:
emb_plot(corona_embs,  plot_path = plot_path , file_name = "Submissions_Corona_PCA",
        label_series = corona_posts["id"], legend = False, plot_labels= False)

In [ ]:
# Explore
#Some more Outise ones
pd.options.display.max_colwidth = 100
print(pn_posts.loc[pn_posts["id"] == "t3_fhtf74","title"].to_string())
print(pn_posts.loc[pn_posts["id"] == "t3_fjubib","title"].to_string())

In [ ]:
#Some more typical ones
print(pn_posts.loc[pn_posts["id"] == "t3_fgj954","title"].to_string())
print(pn_posts.loc[pn_posts["id"] == "t3_fhtymf","title"].to_string())

In [ ]:
#----------------------------#
# 5.) Multidimensional Scaling Tryout
#----------------------------#
# thx to https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html

In [ ]:
from sklearn.manifold import MDS
MDS_embedding = MDS(n_components=2)

In [ ]:
MDS_embedding

In [ ]:
pn_embs.shape

In [ ]:
conservative_embs_transformed = MDS_embedding.fit_transform(conservative_embs.T)

In [ ]:
# This is what goes into the function
conservative_embs_transformed.shape

In [ ]:
conservative_embs.shape

In [ ]:
test = PCA(n_components=2).fit(conservative_embs)
test.components_.shape

In [ ]:
# THis is what goes into the function
test.components_.T.shape

In [ ]:
emb_plot(conservative_embs,  plot_path = plot_path , file_name = "Conservative_PCA",
        label_series = conservative_posts["id"], legend = False, plot_labels= False, MDS = True)

In [ ]:
#CONCLUSION: MDS gives worse almost random results.
# Not here anymore but it once crashed my computer

In [ ]:
#----------------------------#
# 6.) Vlads subreddits check
#----------------------------#

In [ ]:
vlad_subreddits = pd.read_csv(data_path + "subreddits - 2020-04-15.csv")
vlad_subreddits

In [ ]:
subs_grouped = vlad_subreddits.groupby(["category","keep"]).size().reset_index()
subs_grouped

In [ ]:
# GENERAL: 
# 1.) Extract more subs (with SVD approach?)
# 2.) Given this table is complete: +technology, + generic, +ideological
# 3.) Why almost no ideological? It can't be size,atheism, Chapotraphous,  Conservative 
# and (especially!!!)conspiracy would be awesome.
# Especiall conspiracy coupled with networks would be so awesome. 
# I can also try sampling. So we have one more machine
# 4.)

In [ ]:
vlad_subreddits.loc[vlad_subreddits["category"]=="ideological",]

In [ ]:
#----------------------------#
# Appendix 1: SVD/ PCA tests and sources
#----------------------------#
# https://docs.scipy.org/doc/numpy/reference/generated/numpy.linalg.svd.html

In [ ]:
#Intro
a = np.random.randn(9, 6) + 1j*np.random.randn(9, 6)
b = np.random.randn(2, 7, 8, 3) + 1j*np.random.randn(2, 7, 8, 3)

In [ ]:
u, s, vh = np.linalg.svd(a, full_matrices=True)
u.shape, s.shape, vh.shape

In [ ]:
 u, s, vh = np.linalg.svd(a, full_matrices=False)
u.shape, s.shape, vh.shape
((9, 6), (6,), (6, 6))

In [ ]:
# Intro 2:
# thx a bunch to https://towardsdatascience.com/entity-embedding-using-pca-and-kernel-pca-798b8b2e8c2f

In [ ]:
import numpy as np
import pandas as pd
np.set_printoptions(linewidth=200)
random_data = np.random.random((1000,7))
#The next line makes weekend much more likely
random_data *= np.array([1,1,1,1,1,6,6]).reshape(1, -1) * random_data
one_hot_encoded = (random_data == random_data.max(axis=1).reshape(-1, 1)).astype('int')
one_hot_encoded.sum(axis=0)

In [ ]:
one_hot_encoded

In [ ]:
one_hot_encoded.shape

In [ ]:
from sklearn.decomposition import PCA
model = PCA(n_components=2).fit(one_hot_encoded)

In [ ]:
model

In [ ]:
%matplotlib inline
import seaborn as sns
category_vectors = model.components_.T
category_vector_frame=pd.DataFrame(category_vectors, index=['mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun'], 
                                   columns=['col1', 'col2']).reset_index()
sns.scatterplot(data=category_vector_frame, x='col1', y='col2', hue='index');

In [ ]:
category_vectors

In [ ]:
category_vector_frame

In [ ]:
#----------------------------#
# 5.) Vlads subreddit
#----------------------------#

In [ ]:
vlad_subreddits = pd.read_csv(data_path + "subreddits - 2020-04-15.csv")
vlad_subreddits

In [ ]:
subs_grouped = vlad_subreddits.groupby(["category","keep"]).size().reset_index()
subs_grouped

In [ ]:
# GENERAL: 
# 1.) Extract more subs (with SVD approach?)
# 2.) Given this table is complete: +technology, + generic, +ideological
# 3.) Why almost no ideological? It can't be size,atheism, Chapotraphous,  Conservative 
# and (especially!!!)conspiracy would be awesome.
# Especiall conspiracy coupled with networks would be so awesome. 
# I can also try sampling. So we have one more machine
# 4.)

In [ ]:
vlad_subreddits.loc[vlad_subreddits["category"]=="ideological",]